# 03 -- Statistical Feature Analysis

**Corresponds to:** Manuscript Sec.4.3-4.4 (Feature Importance analysis)

We perform univariate statistical testing (Kolmogorov-Smirnov test, Mann-Whitney U test, Cohen's d) on all 408 regional diffusion metrics to identify which individual features best discriminate between groups. This provides biological interpretability independent of the multivariate neural network.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ks_2samp, mannwhitneyu
from tqdm import tqdm
import os, warnings
warnings.filterwarnings('ignore')
np.random.seed(41)

DATA_DIR = os.path.join("data")
FIG_DIR = os.path.join("figures", "statistical_analysis")
os.makedirs(FIG_DIR, exist_ok=True)

plt.rcParams.update({
    'font.family': 'DejaVu Sans Mono', 'font.size': 12,
    'axes.titlesize': 14, 'axes.labelsize': 12
})
COLORS = ['#9671bd', '#77b5b6', '#e08c6c', '#7e7e7e']

## 1. Analysis Function

In [ ]:
def analyze_features(df, diag_col, group1_label, group2_label, output_prefix):
    """Run KS test, Mann-Whitney, and Cohen's d for all features."""
    input_features = [c for c in df.columns if c.startswith("harm_")]

    mask1 = df[diag_col] == group1_label
    mask2 = df[diag_col] == group2_label

    results = []
    for feat in tqdm(input_features, desc=f"Analyzing {output_prefix}"):
        g1 = df.loc[mask1, feat].dropna().values
        g2 = df.loc[mask2, feat].dropna().values
        if len(g1) < 10 or len(g2) < 10:
            continue

        ks_stat, ks_p = ks_2samp(g1, g2)
        mw_stat, mw_p = mannwhitneyu(g1, g2, alternative='two-sided')

        pooled_std = np.sqrt((np.std(g1)**2 + np.std(g2)**2) / 2)
        cohens_d = abs(np.mean(g1) - np.mean(g2)) / pooled_std if pooled_std > 0 else 0

        results.append({
            'feature': feat.replace("harm_", ""),
            'ks_stat': ks_stat, 'ks_pval': ks_p,
            'mw_pval': mw_p, 'cohens_d': cohens_d,
            f'{group1_label}_mean': np.mean(g1),
            f'{group2_label}_mean': np.mean(g2),
            f'{group1_label}_std': np.std(g1),
            f'{group2_label}_std': np.std(g2)
        })

    results_df = pd.DataFrame(results)
    results_df['-log10_ks_p'] = -np.log10(results_df['ks_pval'].clip(lower=1e-300))
    results_df = results_df.sort_values('ks_stat', ascending=False).reset_index(drop=True)

    return results_df

## 2. Patient vs Control Analysis

In [ ]:
df = pd.read_pickle(os.path.join(DATA_DIR, "data_full.pkl"))
print(f"Total samples: {len(df)}")

results_pvc = analyze_features(df, 'diag_pvc', 'Patient', 'Control', 'PvC')
print(f"\nTop 10 features (Patient vs Control):")
print(results_pvc[['feature', 'ks_stat', 'ks_pval', 'cohens_d']].head(10).to_string(index=False))

## 3. SCZ vs Non-SCZ Analysis

In [ ]:
# Filter to patient-only subset
df_patients = df[df['diag_pvc'] == 'Patient'].copy()
print(f"Patient samples for subtype analysis: {len(df_patients)}")

results_scz = analyze_features(df_patients, 'diag_scz', 'SCZ', 'Non SCZ', 'SCZ')
print(f"\nTop 10 features (SCZ vs Non-SCZ):")
print(results_scz[['feature', 'ks_stat', 'ks_pval', 'cohens_d']].head(10).to_string(index=False))

## 4. Feature Distribution Plots

For each task, we plot the top 5 most discriminative features with multi-panel distribution comparisons (histograms, KDE, ECDF, box plots, violin plots).

In [ ]:
def plot_top_features(results, group1, group2, title, filename, n=5):
    top5 = results.head(n)
    n_cols = min(3, n)
    n_rows = int(np.ceil(n / n_cols)) + 1  # +1 for stat summary

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes]

    for i, (_, row) in enumerate(top5.iterrows()):
        ax = axes[i]
        feat = f"harm_{row['feature']}"

        g1 = df.loc[df['diag_pvc'] == group1, feat].dropna().values if 'Patient' in group1 else              df_patients.loc[df_patients['diag_scz'] == group1, feat].dropna().values
        g2 = df.loc[df['diag_pvc'] == group2, feat].dropna().values if 'Patient' in group2 else              df_patients.loc[df_patients['diag_scz'] == group2, feat].dropna().values

        ax.hist(g1, bins=25, alpha=0.6, color=COLORS[0], label=group1, density=True, edgecolor='w')
        ax.hist(g2, bins=25, alpha=0.6, color=COLORS[1], label=group2, density=True, edgecolor='w')
        ax.set_title(row['feature'], fontsize=11)
        ax.legend(fontsize=8)
        ax.set_ylabel('Density')
        ax.grid(True, alpha=0.2)

    # Hide unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(title, fontweight='bold', fontsize=14)
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {filename}")

plot_top_features(results_pvc, 'Patient', 'Control',
                  'Top 5 Discriminative Features: Patient vs Control',
                  os.path.join(FIG_DIR, "top5_patient_vs_control.png"))

if len(results_scz) >= 5:
    plot_top_features(results_scz, 'SCZ', 'Non SCZ',
                      'Top 5 Discriminative Features: SCZ vs Non-SCZ',
                      os.path.join(FIG_DIR, "top5_scz_vs_non_scz.png"))

## 5. Save Results Tables

In [ ]:
results_pvc.to_csv(os.path.join(FIG_DIR, "feature_analysis_pvc.csv"), index=False)
results_scz.to_csv(os.path.join(FIG_DIR, "feature_analysis_scz.csv"), index=False)

print(f"\nSummary:")
print(f"  Patient vs Control -- features with KS p < 0.05: {(results_pvc['ks_pval'] < 0.05).sum()}/{len(results_pvc)}")
print(f"  SCZ vs Non-SCZ -- features with KS p < 0.05: {(results_scz['ks_pval'] < 0.05).sum()}/{len(results_scz)}")
print(f"  Top PvC feature: {results_pvc.iloc[0]['feature']} (KS={results_pvc.iloc[0]['ks_stat']:.4f})")
print(f"  Top SCZ feature: {results_scz.iloc[0]['feature']} (KS={results_scz.iloc[0]['ks_stat']:.4f})")
print("\n✓ Statistical analysis complete.")